In [13]:
from pathlib import Path
import pandas as pd
import torch
import numpy as np
from datetime import datetime
import sys
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/TETHER")

PROJECT_DATA_DIR = Path(
    "/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data"
)
EMBEDDING_SUFFIX = "_protein_embedding.pt"

species = "mm10"

raw_embedding_dir = PROJECT_DATA_DIR / "tf_data" / species / "tf_embeddings_raw"
scaled_embedding_dir = PROJECT_DATA_DIR / "tf_data" / species / "tf_embeddings"
tf_embedding_proj_file = PROJECT_DATA_DIR / "tf_data" / "tf_embedding_projection_d128.pt"

In [3]:
def load_pooled_embeddings(embedding_dir):
    """{TF name (upper) -> mean-pooled embedding} plus each TF's top principal direction.

    Files are unpadded [L, d], one per TF, so the pool is a plain mean over residues.
    """
    embedding_dir = Path(embedding_dir)
    files = sorted(embedding_dir.glob(f"*{EMBEDDING_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"No {EMBEDDING_SUFFIX} files in {embedding_dir}")

    # A directory holding two generations at once is the quietest way for all of this to
    # go wrong: the checks still run, on a mixture, and report a number that is neither
    # generation's. It happened -- a changed output suffix meant new files sat beside old
    # ones instead of replacing them, and the mixture scored ortholog AUROC 0.73 where the
    # new embeddings alone scored 0.99. Timestamps are the cheap tell.
    mtimes = [p.stat().st_mtime for p in files]
    span_hours = (max(mtimes) - min(mtimes)) / 3600
    if span_hours > 24:
        oldest = datetime.fromtimestamp(min(mtimes)).strftime("%Y-%m-%d %H:%M")
        newest = datetime.fromtimestamp(max(mtimes)).strftime("%Y-%m-%d %H:%M")
        print(
            f"\n  WARNING  {embedding_dir} holds files written {span_hours / 24:.1f} days apart\n"
            f"           ({oldest} .. {newest}). If these are two generations, every check\n"
            f"           below is measuring a mixture. Delete the directory and re-project."
        )

    pooled, top_directions = {}, {}
    for path in files:
        name = path.name.replace(EMBEDDING_SUFFIX, "").upper()
        emb = torch.load(path, map_location="cpu", weights_only=True).float().numpy()
        if emb.ndim == 3 and emb.shape[0] == 1:
            emb = emb[0]

        pooled[name] = emb.mean(axis=0)

        centered = emb - emb.mean(axis=0)
        top_directions[name] = np.linalg.svd(centered, full_matrices=False)[2][0]

    return pooled, top_directions

def cosine_matrix(vectors):
    V = np.asarray(vectors, dtype=np.float64)
    norms = np.linalg.norm(V, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    V = V / norms
    return V @ V.T

In [4]:
pooled, top_directions = load_pooled_embeddings(scaled_embedding_dir)

In [10]:
def load_tf_families(species):
    """{TF name (upper) -> CIS-BP Family_Name} for one species."""
    info_path = (
        PROJECT_DATA_DIR / "databases" / "motif_information" / species
        / "TF_Information_all_motifs.txt"
    )
    if not info_path.exists():
        return None

    info = pd.read_csv(info_path, sep="\t", header=0, low_memory=False)
    info["TF_Name"] = info["TF_Name"].astype(str).str.upper()
    info = info.dropna(subset=["Family_Name"])
    return info.drop_duplicates("TF_Name").set_index("TF_Name")["Family_Name"].to_dict()

families = load_tf_families("mm10")


In [14]:


def load_cisbp_motifs(motif_dir, tf_names, direct_only=True, min_ic=0.25, min_len=4):
    """One representative PWM per TF by highest information content (IC)

    direct_only keeps TF_Status == "D". CIS-BP "I" motifs are inferred by copying
    the motif of a TF with a similar DNA-binding domain.
    """
    motif_dir = Path(motif_dir)
    info = pd.read_csv(motif_dir / "TF_Information_all_motifs.txt", sep="\t")
    info["TF_Name"] = info["TF_Name"].str.upper()
    info = info[info["Motif_ID"].notna() & (info["Motif_ID"] != ".")]
    if direct_only:
        info = info[info["TF_Status"] == "D"]
    info = info[info["TF_Name"].isin(set(tf_names))]
    
    def pwm_information_content(pwm):
        """Per-column information content in bits (0 = uniform, 2 = a fixed base)."""
        p = np.clip(pwm, 1e-9, None)
        return 2.0 + (p * np.log2(p)).sum(axis=1)


    def trim_pwm(pwm, min_ic=0.25):
        """Drop uninformative leading/trailing columns.

        Near-uniform flanks make every motif resemble every other motif, because all
        near-uniform columns resemble each other.
        """
        ic = pwm_information_content(pwm)
        keep = np.flatnonzero(ic >= min_ic)
        return pwm if len(keep) == 0 else pwm[keep[0] : keep[-1] + 1]

    pwms, families = {}, {}
    for tf_name, rows in info.groupby("TF_Name"):
        best, best_ic = None, -np.inf
        for motif_id in rows["Motif_ID"].unique():
            path = motif_dir / "pwms_all_motifs" / f"{motif_id}.txt"
            if not path.exists():
                continue
            mat = pd.read_csv(path, sep="\t", index_col=0).values.astype(float)
            if mat.ndim != 2 or mat.shape[1] != 4 or mat.shape[0] < min_len:
                continue
            mat = mat / np.clip(mat.sum(axis=1, keepdims=True), 1e-9, None)
            mat = trim_pwm(mat, min_ic=min_ic)
            total_ic = pwm_information_content(mat).sum()
            if total_ic > best_ic:
                best, best_ic = mat, total_ic
        if best is not None and len(best) >= min_len:
            pwms[tf_name] = best
            families[tf_name] = rows["Family_Name"].iloc[0]
    return pwms, families

TF_MOTIF_DIR = PROJECT_DIR.parent / "data" / "databases" / "motif_information" / species

tf_names = pooled.keys()
tf_cisbp_pwms, tf_families = load_cisbp_motifs(TF_MOTIF_DIR, tf_names, direct_only=True, min_ic=0.25, min_len=4)


In [ ]:
import umap

# Stack the pooled embeddings into the format expected by UMAP
tf_names = sorted(pooled)
X = np.stack([pooled[tf] for tf in tf_names]).astype(np.float32)

if not np.isfinite(X).all():
    raise ValueError("Some pooled embeddings contain NaN or infinity")

# Project the pooled embeddings into a 3D space using UMAP
coords = umap.UMAP(
    n_components=2,
    n_neighbors=min(15, len(tf_names) - 1),
    min_dist=0.1,
    metric="cosine",
    random_state=42,
).fit_transform(X)

umap_df = pd.DataFrame({
    "TF": tf_names,
    "UMAP1": coords[:, 0],
    "UMAP2": coords[:, 1],
})

umap_df["Family"] = umap_df["TF"].map(tf_families)

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
import plotly.express as px

fig = px.scatter(
    umap_df,
    x="UMAP1",
    y="UMAP2",
    color="Family",
    hover_name="TF",
    opacity=0.8,
    title="Mean-pooled TF embeddings",
    width=750,
    height=600,
)

# Remove the background color (makes it transparent)
fig.update_layout(
    plot_bgcolor="rgba(0, 0, 0, 0)",
)

# Add border lines by updating the axes
fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True)
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True)

# Change point size
fig.update_traces(marker=dict(size=5))
fig.show()

In [39]:
from sklearn.neighbors import NearestNeighbors

labels = pd.Series(tf_names).map(families).fillna("Unknown").to_numpy()
k = min(10, len(tf_names) - 1)

neighbors = NearestNeighbors(
    n_neighbors=k + 1, metric="cosine"
).fit(X).kneighbors(X, return_distance=False)[:, 1:]

valid = labels != "Unknown"
family_counts = pd.Series(labels[valid]).value_counts()
valid &= pd.Series(labels).map(family_counts).fillna(0).to_numpy() >= 2

observed = (labels[neighbors[valid]] == labels[valid, None]).mean()
expected = np.mean(
    (family_counts.loc[labels[valid]].to_numpy() - 1) / (len(labels) - 1)
)

print(f"Same-family neighbors: {observed:.1%}")
print(f"Random-label expectation: {expected:.1%}")

Same-family neighbors: 47.4%
Random-label expectation: 7.8%
